# 03 — Statistical Tests: Merit-Order Effect & Negative-Price Drivers

**Tests in this notebook:**
1. Mann-Whitney U test with Cliff's delta and bootstrap 95% CIs — high vs low wind/solar/residual-load groups
2. Hour-stratified tests controlled for diurnal demand, with Benjamini-Hochberg FDR correction
3. Chi-squared / Fisher's Exact test + Odds Ratio for negative-price occurrence in high-solar and high-wind hours
4. OLS regression with Newey-West HAC robust standard errors + Ljung-Box residual diagnostics
5. Stated limitations honestly

> **Pre-defined hypotheses:** All group definitions (top/bottom quartile cutoffs, hour stratification) were fixed before running any tests. No fishing for significant p-values.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import acorr_ljungbox
from src.stats_tests import (
    test_generation_regimes, stratified_hourly_tests,
    test_negative_price_odds, fit_hac_merit_order_regression
)

df = pd.read_parquet(ROOT / 'data' / 'processed' / 'merged_hourly.parquet')
print(f'Loaded {len(df):,} rows')

Loaded 69,023 rows


## 1. Non-Parametric Tests: High vs Low Generation Regimes

**Why Mann-Whitney U?** Electricity prices are heavily right-skewed (skewness = 2.36, kurtosis = 8.74). The t-test assumes approximate normality of group means — fragile here because extreme values heavily influence the mean. MWU is a rank-based test robust to non-normality.

**Effect size:** Cliff's delta δ ∈ [−1, +1]: |δ| < 0.15 = negligible, 0.15–0.33 = small, 0.33–0.47 = medium, > 0.47 = large.

In [2]:
for col, label in [('wind_total_mw','Total Wind'), ('solar_mw','Solar'), ('residual_load_mw','Residual Load')]:
    r = test_generation_regimes(df, driver_col=col, n_boot=1000)
    print(f'\n--- {label} ---')
    print(f'  Median High: {r["median_high"]:.2f} vs Low: {r["median_low"]:.2f} €/MWh')
    print(f'  Median Diff: {r["median_diff"]:.2f} (95% CI [{r["median_diff_ci_95"][0]:.2f}, {r["median_diff_ci_95"][1]:.2f}])')
    print(f'  p-value: {r["p_value"]:.2e} | Cliff\'s delta: {r["cliffs_delta"]:.4f}')


--- Total Wind ---
  Median High: 54.47 vs Low: 94.61 €/MWh
  Median Diff: -40.14 (95% CI [-41.64, -38.46])
  p-value: 0.00e+00 | Cliff's delta: -0.3800



--- Solar ---
  Median High: 57.85 vs Low: 50.72 €/MWh
  Median Diff: 7.13 (95% CI [5.35, 8.90])
  p-value: 3.91e-18 | Cliff's delta: -0.0535



--- Residual Load ---
  Median High: 111.10 vs Low: 37.36 €/MWh
  Median Diff: 73.74 (95% CI [71.61, 76.23])
  p-value: 0.00e+00 | Cliff's delta: 0.6225


## 2. Hour-Stratified Tests — Controlling for Diurnal Demand

Raw comparison of high-wind vs low-wind prices is confounded by time of day: wind peaks at night when demand is low *and* prices are low. By stratifying within each hour, we control for diurnal demand patterns and isolate the pure wind merit-order effect.

In [3]:
strat = stratified_hourly_tests(df, driver_col='wind_total_mw')
print('Mann-Whitney p-values after Benjamini-Hochberg FDR correction:')
print(strat[['hour','med_diff','cliffs_delta','p_value_fdr','significant_fdr']].to_string(index=False))
print(f'\nAll 24 hours significant after FDR correction: {strat["significant_fdr"].all()}')

Mann-Whitney p-values after Benjamini-Hochberg FDR correction:
 hour  med_diff  cliffs_delta  p_value_fdr  significant_fdr
    0   -55.770     -0.553171 6.460328e-73             True
    1   -56.025     -0.551866 1.172808e-72             True
    2   -56.600     -0.590655 9.629808e-83             True
    3   -58.960     -0.608428 1.467524e-87             True
    4   -60.240     -0.609353 1.467524e-87             True
    5   -58.910     -0.572635 5.905185e-78             True
    6   -52.560     -0.494276 1.048831e-58             True
    7   -43.025     -0.416436 3.175708e-42             True
    8   -32.800     -0.345536 1.397706e-29             True
    9   -26.535     -0.308412 6.142265e-24             True
   10   -30.715     -0.297583 1.956723e-22             True
   11   -28.865     -0.280503 4.008059e-20             True
   12   -26.950     -0.275593 1.691407e-19             True
   13   -24.135     -0.260744 1.240717e-17             True
   14   -24.770     -0.250663 1.98023

## 3. Negative-Price Odds Ratio — High Solar vs High Wind

**Contingency table test:** Does high solar (top quartile of PV generation, restricted to daylight hours 08:00–18:00) significantly increase the odds of observing a negative price?

In [4]:
# Solar: restrict to daylight hours where solar actually generates
df_day = df[(df['hour'] >= 8) & (df['hour'] <= 18)]
res_solar = test_negative_price_odds(df_day, driver_col='solar_mw')
print('=== High Solar → Negative Price ===')
print(f'  Negative rate | High solar: {res_solar["neg_rate_high_gen"]*100:.2f}%')
print(f'  Negative rate | Low solar:  {res_solar["neg_rate_low_gen"]*100:.2f}%')
print(f'  Odds Ratio: {res_solar["odds_ratio"]:.3f} (95% CI: {res_solar["odds_ratio_ci_95"]})')
print(f'  Chi²: {res_solar["chi2_stat"]:.2f}, p={res_solar["p_chi2"]:.2e}')

res_wind = test_negative_price_odds(df, driver_col='wind_total_mw')
print('\n=== High Wind → Negative Price ===')
print(f'  Negative rate | High wind: {res_wind["neg_rate_high_gen"]*100:.2f}%')
print(f'  Negative rate | Low wind:  {res_wind["neg_rate_low_gen"]*100:.2f}%')
print(f'  Odds Ratio: {res_wind["odds_ratio"]:.3f} (95% CI: {res_wind["odds_ratio_ci_95"]})')
print(f'  Chi²: {res_wind["chi2_stat"]:.2f}, p={res_wind["p_chi2"]:.2e}')

=== High Solar → Negative Price ===
  Negative rate | High solar: 18.23%
  Negative rate | Low solar:  2.35%
  Odds Ratio: 9.254 (95% CI: (8.359861907406406, 10.244019451167674))
  Chi²: 2517.23, p=0.00e+00

=== High Wind → Negative Price ===
  Negative rate | High wind: 7.17%
  Negative rate | Low wind:  2.46%
  Odds Ratio: 3.058 (95% CI: (2.8221763265469053, 3.3135037114124795))
  Chi²: 815.07, p=2.86e-179


## 4. Econometric Regression: Merit-Order with HAC Robust Standard Errors

**Model:** `price ~ wind_total_GW + solar_GW + load_GW + C(hour) + C(month) + C(day_of_week) + is_holiday`

**HAC standard errors** (Newey-West, maxlags=24) account for autocorrelation in hourly price residuals, giving valid inference despite violating OLS i.i.d. assumptions.

In [5]:
res = fit_hac_merit_order_regression(df, max_lags=24)
import pandas as pd
coefs = pd.DataFrame({
    'Coefficient': res.params[['wind_total_gw','solar_gw','load_gw']],
    'HAC Std Err': res.bse[['wind_total_gw','solar_gw','load_gw']],
    't-stat': res.tvalues[['wind_total_gw','solar_gw','load_gw']],
    'p-value': res.pvalues[['wind_total_gw','solar_gw','load_gw']],
    '95% CI Lo': res.conf_int()[0][['wind_total_gw','solar_gw','load_gw']],
    '95% CI Hi': res.conf_int()[1][['wind_total_gw','solar_gw','load_gw']]
})
print(f'R² = {res.rsquared:.4f} | Adj R² = {res.rsquared_adj:.4f} | N = {int(res.nobs):,}')
coefs.round(4)

R² = 0.2151 | Adj R² = 0.2146 | N = 68,890


,Coefficient,HAC Std Err,t-stat,p-value,95% CI Lo,95% CI Hi
wind_total_gw,-2.8598,0.1171,-24.4271,0.0,-3.0893,-2.6304
solar_gw,-2.3531,0.1072,-21.9549,0.0,-2.5632,-2.1431
load_gw,1.3710,0.2389,5.7386,0.0,0.9028,1.8393


In [6]:
# Residual diagnostics — Ljung-Box test for autocorrelation
lb = acorr_ljungbox(res.resid, lags=[1, 12, 24], return_df=True)
print('Ljung-Box test for autocorrelation in OLS residuals:')
print(lb)
print('\nNOTE: Significant autocorrelation is expected in hourly price residuals.')
print('HAC standard errors correct for this in inference; residuals do NOT need to be white noise for HAC to be valid.')

Ljung-Box test for autocorrelation in OLS residuals:
         lb_stat  lb_pvalue
1   6.639861e+04        0.0
12  6.652271e+05        0.0
24  1.289271e+06        0.0

NOTE: Significant autocorrelation is expected in hourly price residuals.
HAC standard errors correct for this in inference; residuals do NOT need to be white noise for HAC to be valid.


## 5. Limitations (Stated Honestly)

1. **Correlation ≠ causation.** Wind and solar generation are correlated with time of day, temperature and season — all of which also affect demand and thus prices. Even with calendar controls, residual confounding exists.
2. **Omitted variables.** Natural gas prices (the primary marginal cost setter), carbon (EUA) prices, and cross-border import flows are absent from this dataset. The 2021–2022 price spike is explained primarily by the global gas crisis, not captured by renewable generation alone.
3. **Autocorrelation.** The Ljung-Box test confirms strong autocorrelation at all tested lags (p ≈ 0). HAC inference is valid, but time-series models (VAR, ARIMAX) would be more appropriate for causal analysis.
4. **Multiple testing.** 24 hour-stratified tests were performed. Benjamini-Hochberg FDR correction was applied; all 24 remained significant, reducing (but not eliminating) the multiple-comparisons concern.
5. **Nuclear data.** Nuclear generation is structurally missing from 2024 (Atomausstieg). This variable was excluded from models.